# Pre-processing with Spark

### Dataset: `iris.csv` (150 righe con 50 esemplari ciascuna)
> **Nota metodologica**: Questo progetto ha uno scopo esclusivamente didattico. Per un dataset così piccolo come quello di iris.csv, l'utilizzo di un motore distribuito come Spark è tecnicamente sproporzionato.

> Questo esercizio ha l'obiettivo di offrire un apprendimento pratico e approfondito degli strumenti e della logica del framework Spark. Fornisce inoltre un'introduzione al Machine Learning in PySpark per esplorare i piani di esecuzione fisici e comprendere i concetti architetturali chiave.

## **0.1 Setup dell'ambiente**

In [3]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Su Windows: decommenta e adatta se winutils non e' gia' nelle variabili di sistema
# os.environ["HADOOP_HOME"] = r"C:\hadoop"
# os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

print("Interprete    :", sys.executable)
print("HADOOP_HOME   :", os.environ.get("HADOOP_HOME", "NON IMPOSTATO"))
print("SPARK_LOCAL_IP:", os.environ.get("SPARK_LOCAL_IP"))

Interprete    : C:\Users\Allysa\Documents\MachineLearning-in-Spark\.venv\Scripts\python.exe
HADOOP_HOME   : C:\hadoop
SPARK_LOCAL_IP: 127.0.0.1


## **1. Avvio della sessione Spark**

In [4]:
from pyspark.sql import SparkSession

spark = (SparkSession
    .builder
    .appName("IrisPreProcessing")
    .config("master", "local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.python,authenticate.socketTimeout", "60")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.executor.memory", "4g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print("Spark avviato:", spark.version)

Spark avviato: 4.2.0


In [5]:
smoke_test_canale = spark.sparkContext.parallelize([1, 2, 3]).map(lambda x: x * 2).collect()

if smoke_test_canale == [2, 4, 6]:
    print("Test superato.")
else:
    raise ValueError("Errore nella comucazione JVM driver <-> Python worker")

Test superato.


## **2. Caricamento dati**

In [10]:
DATA_PATH = Path("data/iris.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"File non esiste nel percorso specificato")

In [12]:
#Con schema implicito
df = spark.read.csv(str(DATA_PATH), header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows


In [20]:
#Con schema esplicito (User-defined)
from pyspark.sql.types import StructField, StructType, StringType, DoubleType

column_mapping = {
    "sepal_length": DoubleType(),
    "sepal_width": DoubleType(),
    "petal_length": DoubleType(),
    "petal_width": DoubleType(),
    "species": StringType(),
}

SCHEMA = StructType([
    StructField(name, dtype, True) for name, dtype in column_mapping.items()
])

df = spark.read.option("header", "true").csv(str(DATA_PATH), schema=SCHEMA).cache()

df.show(5)

#Conteggio per specie
df.groupBy("species").count().show()

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows

Conteggio per specie
+----------+-----+
|   species|count|
+----------+-----+
|versicolor|   50|
| virginica|   50|
|    setosa|   50|
+----------+-----+



## **3. Statistiche descrittive**

> Le colonne derivate `sepal_area` e `petal_area` sono state create esclusivamente come feature sintetiche. Non esiste un calcolo analogo per i sepali o i petali.

In [70]:
import pyspark.sql.functions as F

df_area = (
    df.withColumn(("sepal_area"), F.col("sepal_length") * F.col("sepal_width"))
      .withColumn(("petal_area"), F.col("petal_length") * F.col("petal_width"))
)

stats_sepal= df_area.agg(
    F.round(F.mean("sepal_length")).alias("media_length"),
    F.round(F.mean("sepal_width")).alias("media_width"),
    F.round(F.mean("sepal_area")).alias("media_area"),
    F.min("sepal_area").alias("min_area"),
    F.max("sepal_area").alias("max_area")
)

stats_petal= df_area.agg(
    F.round(F.mean("petal_length")).alias("media_length"),
    F.round(F.mean("petal_width")).alias("media_width"),
    F.round(F.mean("petal_area")).alias("media_area"),
    F.round(F.min("petal_area")).alias("min_area"),
    F.max("petal_area").alias("max_area")
)

print("======================== SEPAL ========================")
stats_sepal.show()
print("======================== PETAL ========================")
stats_petal.show()

======================== SEPAL ========================
+------------+-----------+----------+--------+--------+
|media_length|media_width|media_area|min_area|max_area|
+------------+-----------+----------+--------+--------+
|         6.0|        3.0|      18.0|    10.0|   30.02|
+------------+-----------+----------+--------+--------+

======================== PETAL ========================
+------------+-----------+----------+--------+--------+
|media_length|media_width|media_area|min_area|max_area|
+------------+-----------+----------+--------+--------+
|         4.0|        1.0|       6.0|     0.0|   15.87|
+------------+-----------+----------+--------+--------+



In [79]:
#Operazioni filtro sepal/petal area > sepal/petal area media

val_scalare_sepal = df_area.select(F.mean("sepal_area")).collect()[0][0]
sepal_filtro = df_area.filter(F.col("sepal_area") > val_scalare_sepal)

val_scalare_petal = df_area.select(F.mean("petal_area")).collect()[0][0]
petal_filtro = df_area.filter(F.col("petal_area") > val_scalare_sepal)

print(f"Sepal area > media: {sepal_filtro.count()}")
print(f"Petal area > media: {petal_filtro.count()}")

Sepal area > media: 34
Petal area > media: 0


## **4. Codifica della colonna `species`**